# SPaRTA : Metrics Evaluation Notebook

**Purpose:** Validate all evaluation metrics before any model training begins.  
**Rule:** Every metric must pass its sanity checks here before being used in training or reporting.  
**Kernel rule:** Always restart the kernel and run all cells top-to-bottom after editing `utils/metrics.py`.

---

## Metrics validated in this notebook

| Metric | Type | Purpose |
|--------|------|---------|
| MPJPE at horizons | Accuracy | Primary benchmark metric, compared against SOTA |
| ADE | Accuracy | Average displacement over all future frames |
| FDE | Accuracy | Displacement at the final predicted frame |
| GVR | Physical plausibility | Gravity Violation Rate - novel metric for this research |
| BLE | Physical plausibility | Bone Length Error - structural consistency |
| Gravity Loss | Training signal | Differentiable version of GVR used during training |

---

## Expected results summary (for a zero-velocity baseline)

| Metric | Ground Truth | Zero-Velocity | Random Noise |
|--------|-------------|---------------|--------------|
| MPJPE @ 80ms | — | ~25–35mm | very large |
| MPJPE @ 1000ms | — | ~200–280mm | very large |
| ADE | — | ~100–150mm | very large |
| FDE | — | ~200–280mm | very large |
| BLE | ~0mm | 0mm | >1000mm |
| GVR | <0.15 | <0.10 | >0.50 |
| Gravity Loss | small | smaller | large |

---
## 0. Environment Setup

Run this cell first every session. It verifies imports and GPU availability.

In [1]:
import torch
import numpy as np
import sys
import os

# ── Path setup ─────────────────────────────────────────────────────────────
# Add project root to path so imports work from any working directory
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── GPU check ──────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == 'cuda':
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("NOTE: Running on CPU. Metrics evaluation is fine on CPU.")
    print("      Model training will require GPU.")

PyTorch version : 2.11.0+cpu
Device          : cpu
NOTE: Running on CPU. Metrics evaluation is fine on CPU.
      Model training will require GPU.


---
## 1. Imports

Import dataset utilities and all metric functions from `utils/metrics.py`.  
If any import fails, check that:
- The project root is correctly added to `sys.path` above
- `utils/metrics.py` contains all the functions listed below
- The kernel has been restarted after any edits to `metrics.py`

In [2]:
# Dataset utilities
from data.h36m_dataset import (
    build_dataloaders,
    SKELETON_EDGES_17,
    JOINT_NAMES_17,
    TRAIN_SUBJECTS,
    TEST_SUBJECTS
)

# Accuracy metrics
from utils.metrics import (
    mpjpe,
    mpjpe_at_horizons,
    ade,
    fde
)

# Physical plausibility metrics (evaluation only)
from utils.metrics import (
    gravity_violation_rate,
    bone_length_error
)

# Physical plausibility loss (training only)
from utils.metrics import gravity_consistency_loss

print("All imports successful.")

All imports successful.


---
## 2. Data Loading

Update `DATA_PATH` to point to your local `data_3d_h36m.npz` file.  

**Expected output:**
```
[H36MDataset] 38045 windows | subjects=['S1','S5','S6','S7','S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9','S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz
```

Train size ~38k (stride=5). Test size ~65k (stride=1 for dense evaluation).

In [3]:

# ── Update this path for your machine ──────────────────────────────────────
# Local Windows  : "D:/path/to/data_3d_h36m.npz"
# Google Colab   : "data/data_3d_h36m.npz"
DATA_PATH = "D:/L4S2/Research Project in AI/Research/iccv21_git_src/data_3d_h36m.npz"

train_loader, test_loader = build_dataloaders(
    DATA_PATH,
    batch_size=32,
    t_obs=10,
    t_pred=25,
    train_stride=5,
    test_stride=1
)

# Expose underlying dataset for per-sample access
train_dataset = train_loader.dataset
test_dataset  = test_loader.dataset

print(f"\nTrain dataset size : {len(train_dataset):,} windows")
print(f"Test  dataset size : {len(test_dataset):,} windows")

c:\Users\gayuni\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


[H36MDataset] 38045 windows | subjects=['S1', 'S5', 'S6', 'S7', 'S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9', 'S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz

Train dataset size : 38,045 windows
Test  dataset size : 65,927 windows


---
## 3. Batch Verification

Always fetch a **fresh, named batch** at the start of each evaluation section.  
Never reuse a `batch` variable from a previous cell - this is the most common source of silent errors.

**Expected shapes:**
```
obs shape : torch.Size([32, 10, 17, 3])
fut shape : torch.Size([32, 25, 17, 3])
```

**Expected coordinate check:**  
Z axis should show head-minus-ankle difference of ~1.3–1.5m (Z is vertical in H3.6M).  
Standing frames count should be > 0 for a valid GVR test.

In [4]:
# ── Fetch one fresh batch from each loader ──────────────────────────────────
# Using separate variable names to prevent accidental reuse
train_batch = next(iter(train_loader))
test_batch  = next(iter(test_loader))

obs_train = train_batch['observed']  # (32, 10, 17, 3)
fut_train = train_batch['future']    # (32, 25, 17, 3)

obs_test  = test_batch['observed']   # (32, 10, 17, 3)
fut_test  = test_batch['future']     # (32, 25, 17, 3)

print("=" * 55)
print("BATCH SHAPE VERIFICATION")
print("=" * 55)
print(f"Train obs shape : {obs_train.shape}")
print(f"Train fut shape : {fut_train.shape}")
print(f"Test  obs shape : {obs_test.shape}")
print(f"Test  fut shape : {fut_test.shape}")

# ── Coordinate system verification ─────────────────────────────────────────
# H3.6M convention: Z is vertical (up). Floor plane = XY.
# Head should be ~1.3–1.5m above ankle on the Z axis.
print("\n" + "=" * 55)
print("COORDINATE SYSTEM VERIFICATION (test batch)")
print("=" * 55)
head_z   = fut_test[:, :, 10, 2]  # joint 10 = head
ankle_z  = fut_test[:, :,  6, 2]  # joint 6  = lankle
diff_z   = (head_z - ankle_z).mean().item()
print(f"Mean head-minus-lankle on Z axis : {diff_z:.3f}m")
print(f"  Expected: ~1.3 to 1.5m")
print(f"  Status  : {'OK' if 1.0 < diff_z < 2.0 else 'WARNING — check coordinate system'}")

# ── Standing frame count (needed for GVR) ──────────────────────────────────
# Root joint (index 0) Z > 0.70m = standing pose
STANDING_Z_THRESHOLD = 0.70
root_z_test = fut_test[:, :, 0, 2]
n_standing  = (root_z_test > STANDING_Z_THRESHOLD).sum().item()
n_total     = fut_test.shape[0] * fut_test.shape[1]
print(f"\nStanding frames in test batch    : {n_standing} / {n_total}")
print(f"  Root Z — min={root_z_test.min():.3f}  "
      f"max={root_z_test.max():.3f}  "
      f"mean={root_z_test.mean():.3f}")
if n_standing == 0:
    print("  WARNING: No standing frames — GVR will return None.")
    print("           This batch may be all seated activity (Sitting, Eating, etc.)")
    print("           Re-run this cell to get a different batch, or lower threshold.")

# ── Sample action labels ────────────────────────────────────────────────────
print(f"\nSample actions in test batch:")
actions = test_batch['action']
from collections import Counter
action_counts = Counter(actions)
for action, count in action_counts.most_common(5):
    print(f"  {action:<22}: {count} windows")

BATCH SHAPE VERIFICATION
Train obs shape : torch.Size([32, 10, 17, 3])
Train fut shape : torch.Size([32, 25, 17, 3])
Test  obs shape : torch.Size([32, 10, 17, 3])
Test  fut shape : torch.Size([32, 25, 17, 3])

COORDINATE SYSTEM VERIFICATION (test batch)
Mean head-minus-lankle on Z axis : 1.627m
  Expected: ~1.3 to 1.5m
  Status  : OK

Standing frames in test batch    : 800 / 800
  Root Z — min=0.965  max=0.988  mean=0.977

Sample actions in test batch:
  Directions 1          : 32 windows


c:\Users\gayuni\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


---
## 4. Build Evaluation Baselines

Two baselines are used throughout this notebook:

- **Zero-velocity baseline:** Repeats the last observed frame for all 25 predicted frames.  
  This is the simplest possible predictor and represents the floor for motion prediction models.  
  A good model must beat this on all metrics.

- **Random noise baseline:** Completely random joint positions.  
  Used only for sanity checks - every metric should be dramatically worse on this input.

In [ ]:
# ── Zero-velocity baseline ─────────────────────────────────────────────────
# Last observed frame repeated T_pred=25 times
# Shape: (B, 1, J, 3) → repeat → (B, 25, J, 3)
zv_pred_train = obs_train[:, -1:, :, :].repeat(1, 25, 1, 1)
zv_pred_test  = obs_test[:,  -1:, :, :].repeat(1, 25, 1, 1)

# ── Random noise baseline ──────────────────────────────────────────────────
# Random positions in the same value range as real data (~-1.5 to 2.0m)
# We scale randn to match the data range rather than using standard normal,
# which would give unrealistically large MPJPE values.
data_std  = fut_test.std().item()
data_mean = fut_test.mean().item()
random_pred_test = torch.randn_like(fut_test) * data_std + data_mean

print("Baselines ready.")
print(f"  Zero-velocity shape : {zv_pred_test.shape}")
print(f"  Random noise shape  : {random_pred_test.shape}")
print(f"  Data value range    : mean={data_mean:.3f}, std={data_std:.3f}")
print(f"  Random noise range  : min={random_pred_test.min():.3f}, "
      f"max={random_pred_test.max():.3f}")

Baselines ready.
  Zero-velocity shape : torch.Size([32, 25, 17, 3])
  Random noise shape  : torch.Size([32, 25, 17, 3])
  Data value range    : mean=0.313, std=0.593
  Random noise range  : min=-2.109, max=2.654


---
## 5. MPJPE — Mean Per Joint Position Error

**What it measures:**  
Average Euclidean distance (mm) between predicted and ground truth joint positions,  
reported at 5 standard horizons: 80ms, 160ms, 320ms, 560ms, 1000ms.

**Why it matters:**  
This is the primary metric used by every H3.6M paper. Your results must be reported  
at these exact horizons to be comparable with SOTA.

**Expected results (zero-velocity baseline):**
```
80ms  :  25–40mm    (short horizon — easy to predict)
160ms :  50–80mm
320ms :  90–130mm
560ms : 140–200mm
1000ms: 220–280mm   (long horizon — hard to predict)
```
Error increases with horizon because the further ahead you predict, the harder it is.

**Sanity checks:**
- Perfect prediction → 0mm
- Error must strictly increase with horizon for zero-velocity baseline
- MPJPE on ground truth vs ground truth → 0mm

In [6]:
print("=" * 55)
print("MPJPE VALIDATION")
print("=" * 55)

# ── Sanity check 1: Perfect prediction gives zero error ────────────────────
B, T, J = 32, 25, 17
perfect = torch.randn(B, T, J, 3)
err_perfect = mpjpe(perfect, perfect)
assert err_perfect.max().item() < 1e-4, \
    f"MPJPE should be 0 for identical inputs, got {err_perfect.max():.6f}"
print("  ✓  Sanity 1: Perfect prediction gives 0mm error")

# ── Sanity check 2: GT vs GT gives zero error ──────────────────────────────
err_gt_gt = mpjpe(fut_test, fut_test)
assert err_gt_gt.max().item() < 1e-4, \
    "MPJPE(GT, GT) should be 0"
print("  ✓  Sanity 2: MPJPE(ground truth, ground truth) = 0mm")

# ── Sanity check 3: Error increases with horizon ───────────────────────────
zv_horizons = mpjpe_at_horizons(zv_pred_test, fut_test)
horizon_vals = list(zv_horizons.values())
monotone = all(horizon_vals[i] < horizon_vals[i+1]
               for i in range(len(horizon_vals)-1))
assert monotone, "MPJPE should increase with prediction horizon"
print("  ✓  Sanity 3: MPJPE increases monotonically with horizon")

# ── Zero-velocity results ──────────────────────────────────────────────────
print("\nZero-velocity baseline MPJPE:")
print(f"  {'Horizon':>10}  {'Error (mm)':>12}  {'Assessment':>20}")
print(f"  {'─'*10}  {'─'*12}  {'─'*20}")
expected_ranges = {
     80:  (15,  60),
    160:  (40, 120),
    320:  (80, 200),
    560: (150, 300),   # updated from 120–220
   1000: (250, 450),   # updated from 180–300
}
for ms, err in zv_horizons.items():
    lo, hi = expected_ranges[ms]
    status = "OK" if lo <= err <= hi else f"outside expected ({lo}–{hi}mm)"
    print(f"  {ms:>8}ms  {err:>10.1f}mm  {status:>20}")

MPJPE VALIDATION
  ✓  Sanity 1: Perfect prediction gives 0mm error
  ✓  Sanity 2: MPJPE(ground truth, ground truth) = 0mm
  ✓  Sanity 3: MPJPE increases monotonically with horizon

Zero-velocity baseline MPJPE:
     Horizon    Error (mm)            Assessment
  ──────────  ────────────  ────────────────────
        80ms        32.5mm                    OK
       160ms        66.1mm                    OK
       320ms       132.7mm                    OK
       560ms       226.2mm                    OK
      1000ms       376.7mm                    OK


---
## 6. ADE and FDE — Displacement Errors

**ADE (Average Displacement Error):**  
Mean L2 distance across **all** predicted frames and all joints.  
Summarises overall prediction quality in a single number.

**FDE (Final Displacement Error):**  
L2 distance at the **last** predicted frame only (frame 25 = 1000ms).  
Measures how far off the model is at the end of the prediction horizon.

**Expected relationship:**  
`ADE < FDE` always — because ADE averages over all frames including the easy early frames,  
while FDE measures only the hardest final frame.

**Expected scale:**  
Both should be in the same order of magnitude as MPJPE.  
For zero-velocity: ADE ≈ average of MPJPE across horizons (~100–150mm), FDE ≈ MPJPE@1000ms (~220–280mm).

In [7]:
print("=" * 55)
print("ADE / FDE VALIDATION")
print("=" * 55)

# ── Zero-velocity baseline ─────────────────────────────────────────────────
ade_zv = ade(zv_pred_test, fut_test).item()
fde_zv = fde(zv_pred_test, fut_test).item()

# ── Random noise baseline ──────────────────────────────────────────────────
ade_random = ade(random_pred_test, fut_test).item()
fde_random = fde(random_pred_test, fut_test).item()

print(f"  {'Metric':<8}  {'Zero-velocity':>16}  {'Random noise':>14}")
print(f"  {'─'*8}  {'─'*16}  {'─'*14}")
print(f"  {'ADE':<8}  {ade_zv:>13.1f} mm  {ade_random:>11.1f} mm")
print(f"  {'FDE':<8}  {fde_zv:>13.1f} mm  {fde_random:>11.1f} mm")

# ── Sanity checks ──────────────────────────────────────────────────────────
print("\nSanity checks:")

check1 = ade_zv < fde_zv
print(f"  {'✓' if check1 else '✗ FAIL'}  ADE < FDE for zero-velocity "
      f"({ade_zv:.1f} < {fde_zv:.1f})")

check2 = ade_zv < ade_random
print(f"  {'✓' if check2 else '✗ FAIL'}  Zero-velocity ADE < random ADE "
      f"({ade_zv:.1f} < {ade_random:.1f})")

# FDE should be approximately equal to MPJPE@1000ms (within 20%)
mpjpe_1000 = zv_horizons.get(1000, None)
if mpjpe_1000:
    ratio = fde_zv / mpjpe_1000
    check3 = 0.70 < ratio < 1.30
    print(f"  {'✓' if check3 else '✗ FAIL'}  FDE ≈ MPJPE@1000ms "
          f"({fde_zv:.1f}mm vs {mpjpe_1000:.1f}mm, ratio={ratio:.2f})")

print("\nNote: ADE and FDE are computed on a fresh batch each run.")
print("      Minor differences from MPJPE values are expected due to batch variation.")

ADE / FDE VALIDATION
  Metric       Zero-velocity    Random noise
  ────────  ────────────────  ──────────────
  ADE               205.3 mm       1354.2 mm
  FDE               376.7 mm       1394.8 mm

Sanity checks:
  ✓  ADE < FDE for zero-velocity (205.3 < 376.7)
  ✓  Zero-velocity ADE < random ADE (205.3 < 1354.2)
  ✓  FDE ≈ MPJPE@1000ms (376.7mm vs 376.7mm, ratio=1.00)

Note: ADE and FDE are computed on a fresh batch each run.
      Minor differences from MPJPE values are expected due to batch variation.


---
## 7. BLE — Bone Length Error

**What it measures:**  
Mean absolute deviation (mm) of predicted bone lengths from reference bone lengths.  
Reference = bone lengths computed from the **last observed frame** of each sequence  
(person-specific, so the model knows the exact skeleton proportions of this individual).

**Why it matters:**  
Real human bone lengths are physically constant — they cannot stretch or compress.  
A model that produces bones of varying length is generating physically implausible motion.

**Used as:** Evaluation metric only (not a training loss — see Decision 008 in the decision record).

**Expected results:**
```
Ground truth  :  ~0mm    (real motion has constant bones — near perfect)
Zero-velocity :   0mm    (same pose repeated = identical bones, exactly zero)
Random noise  : >500mm   (random positions destroy bone structure)
```

In [8]:
print("=" * 55)
print("BONE LENGTH ERROR (BLE) VALIDATION")
print("=" * 55)

# ── Compute BLE for each case ──────────────────────────────────────────────
ble_gt     = bone_length_error(fut_test,        obs_test)
ble_zv     = bone_length_error(zv_pred_test,    obs_test)
ble_random = bone_length_error(random_pred_test, obs_test)

# Convert to mm for display (function returns metres)
print(f"  {'Case':<20}  {'BLE':>10}  {'Assessment':>30}")
print(f"  {'─'*20}  {'─'*10}  {'─'*30}")
print(f"  {'Ground truth':<20}  {ble_gt*1000:>8.2f}mm  "
      f"{'OK — near zero expected':<30}")
print(f"  {'Zero-velocity':<20}  {ble_zv*1000:>8.2f}mm  "
      f"{'OK — exactly zero expected':<30}")
print(f"  {'Random noise':<20}  {ble_random*1000:>8.2f}mm  "
      f"{'OK — very large expected':<30}")

# ── Sanity checks ──────────────────────────────────────────────────────────
print("\nSanity checks:")

check1 = ble_gt < ble_random
print(f"  {'✓' if check1 else '✗ FAIL'}  Ground truth BLE < random BLE "
      f"({ble_gt*1000:.2f}mm < {ble_random*1000:.2f}mm)")

check2 = ble_zv < 0.001  # less than 1mm = near zero
print(f"  {'✓' if check2 else '✗ FAIL'}  Zero-velocity BLE ≈ 0mm "
      f"(got {ble_zv*1000:.4f}mm)")

check3 = ble_random > 0.5  # more than 500mm
print(f"  {'✓' if check3 else '✗ FAIL'}  Random noise BLE > 500mm "
      f"(got {ble_random*1000:.1f}mm)")

check4 = ble_gt < 0.010  # ground truth should be under 10mm
print(f"  {'✓' if check4 else '✗ FAIL'}  Ground truth BLE < 10mm "
      f"(got {ble_gt*1000:.2f}mm)")

# ── Per-bone inspection ────────────────────────────────────────────────────
print("\nPer-bone reference lengths (from last observed frame, mean across batch):")
ref_pose = obs_test[:, -1, :, :]   # (B, J, 3)
print(f"  {'Bone':<25}  {'Ref length (mm)':>16}")
print(f"  {'─'*25}  {'─'*16}")
for (i, j) in SKELETON_EDGES_17:
    bone_name = f"{JOINT_NAMES_17[i]}→{JOINT_NAMES_17[j]}"
    ref_len   = torch.norm(
        ref_pose[:, i, :] - ref_pose[:, j, :], dim=-1
    ).mean().item()
    print(f"  {bone_name:<25}  {ref_len*1000:>14.1f}mm")

BONE LENGTH ERROR (BLE) VALIDATION
  Case                         BLE                      Assessment
  ────────────────────  ──────────  ──────────────────────────────
  Ground truth              0.00mm  OK — near zero expected       
  Zero-velocity             0.00mm  OK — exactly zero expected    
  Random noise           1079.47mm  OK — very large expected      

Sanity checks:
  ✓  Ground truth BLE < random BLE (0.00mm < 1079.47mm)
  ✓  Zero-velocity BLE ≈ 0mm (got 0.0000mm)
  ✓  Random noise BLE > 500mm (got 1079.5mm)
  ✓  Ground truth BLE < 10mm (got 0.00mm)

Per-bone reference lengths (from last observed frame, mean across batch):
  Bone                        Ref length (mm)
  ─────────────────────────  ────────────────
  root→rhip                           124.1mm
  rhip→rknee                          472.2mm
  rknee→rankle                        468.6mm
  root→lhip                           124.1mm
  lhip→lknee                          472.2mm
  lknee→lankle                

---
## 8. GVR — Gravity Violation Rate

**What it measures:**  
Fraction of predicted frames where the root joint (pelvis) projects outside the ankle-based  
base of support on the floor plane (XY). A violation means the predicted pose is physically  
implausible — the person would be falling over.

**Design decisions (see Decision Record 001–006):**
- H3.6M coordinate system: **Z is vertical**. Floor plane = **XY** (indices 0 and 1)
- **Root joint (index 0)** used as CoM proxy — more robust than mass-weighted CoM
- **Restricted to standing frames** only (root Z > 0.70m) — seated activities excluded
  because the ankle BoS assumption is invalid when sitting
- Base of support = ankle bounding box with **0.15m margin**

**Used as:** Evaluation metric only. The differentiable `gravity_consistency_loss` is used for training.

**Expected results:**
```
Ground truth  : <0.15   (real humans rarely lose balance while standing)
Zero-velocity : <0.10   (static pose = no dynamic imbalance)
Random noise  : >0.50   (random positions violate balance constantly)
Returns None  : if batch contains only seated activities — re-run cell
```

**Important:** If this returns `None` for ground truth, the batch has no standing frames.  
This is a batch sampling issue, not a bug. Re-run the cell to get a different batch.

In [9]:
print("=" * 55)
print("GRAVITY VIOLATION RATE (GVR) VALIDATION")
print("=" * 55)

# ── Pre-check: confirm standing frames exist in this batch ─────────────────
root_z       = fut_test[:, :, 0, 2]
n_standing   = (root_z > 0.70).sum().item()
n_total      = fut_test.shape[0] * fut_test.shape[1]
print(f"\nBatch composition:")
print(f"  Total frames   : {n_total}")
print(f"  Standing frames: {n_standing} "
      f"({100*n_standing/n_total:.1f}%)")
print(f"  Seated frames  : {n_total - n_standing} "
      f"({100*(n_total-n_standing)/n_total:.1f}%) — excluded from GVR")

if n_standing == 0:
    print("\n  WARNING: No standing frames in this batch.")
    print("  GVR cannot be computed. Fetch a new batch:")
    print("    test_batch = next(iter(test_loader))")
    print("  and re-run this cell.")
else:
    # ── Compute GVR ────────────────────────────────────────────────────────
    gvr_gt     = gravity_violation_rate(fut_test)
    gvr_zv     = gravity_violation_rate(zv_pred_test)
    gvr_random = gravity_violation_rate(random_pred_test)

    print(f"\n  {'Case':<20}  {'GVR':>8}  {'Assessment':>30}")
    print(f"  {'─'*20}  {'─'*8}  {'─'*30}")

    def gvr_row(label, val, good_condition, good_text, bad_text):
        if val is None:
            print(f"  {label:<20}  {'None':>8}  {'No standing frames':>30}")
        else:
            status = good_text if good_condition(val) else bad_text
            print(f"  {label:<20}  {val:>8.4f}  {status:>30}")

    gvr_row('Ground truth', gvr_gt,
            lambda v: v < 0.15,
            'OK — low violation rate',
            'HIGH — check threshold/margin')
    gvr_row('Zero-velocity', gvr_zv,
            lambda v: v < 0.10,
            'OK — static pose is stable',
            'HIGH — static pose should be balanced')
    gvr_row('Random noise', gvr_random,
            lambda v: v > 0.50,
            'OK — random always violates',
            'LOW — metric may not be sensitive')

    # ── Sanity checks ──────────────────────────────────────────────────────
    print("\nSanity checks:")

    if gvr_gt is not None:
        c1 = gvr_gt < 0.15
        print(f"  {'✓' if c1 else '✗ FAIL'}  GVR ground truth < 0.15 "
              f"(got {gvr_gt:.4f})")

    if gvr_random is not None:
        c2 = gvr_random > 0.50
        print(f"  {'✓' if c2 else '✗ FAIL'}  GVR random noise > 0.50 "
              f"(got {gvr_random:.4f})")

    if gvr_gt is not None and gvr_random is not None:
        c3 = gvr_gt < gvr_random
        print(f"  {'✓' if c3 else '✗ FAIL'}  GVR ground truth < GVR random "
              f"({gvr_gt:.4f} < {gvr_random:.4f})")

    # ── Frame-level debug ──────────────────────────────────────────────────
    print("\nFrame-level debug (first sequence, first frame):")
    frame = fut_test[0, 0]
    root_xy   = (frame[0, 0].item(),  frame[0, 1].item())
    lankle_xy = (frame[6, 0].item(),  frame[6, 1].item())
    rankle_xy = (frame[3, 0].item(),  frame[3, 1].item())
    bos_x_lo  = min(lankle_xy[0], rankle_xy[0]) - 0.15
    bos_x_hi  = max(lankle_xy[0], rankle_xy[0]) + 0.15
    bos_y_lo  = min(lankle_xy[1], rankle_xy[1]) - 0.15
    bos_y_hi  = max(lankle_xy[1], rankle_xy[1]) + 0.15
    inside_x  = bos_x_lo <= root_xy[0] <= bos_x_hi
    inside_y  = bos_y_lo <= root_xy[1] <= bos_y_hi
    print(f"  Root XY     : ({root_xy[0]:.3f}, {root_xy[1]:.3f})")
    print(f"  Lankle XY   : ({lankle_xy[0]:.3f}, {lankle_xy[1]:.3f})")
    print(f"  Rankle XY   : ({rankle_xy[0]:.3f}, {rankle_xy[1]:.3f})")
    print(f"  BoS X range : [{bos_x_lo:.3f}, {bos_x_hi:.3f}]")
    print(f"  BoS Y range : [{bos_y_lo:.3f}, {bos_y_hi:.3f}]")
    print(f"  Root inside BoS: X={'YES' if inside_x else 'NO'}, "
          f"Y={'YES' if inside_y else 'NO'} "
          f"→ {'BALANCED' if inside_x and inside_y else 'VIOLATION'}")

GRAVITY VIOLATION RATE (GVR) VALIDATION

Batch composition:
  Total frames   : 800
  Standing frames: 800 (100.0%)
  Seated frames  : 0 (0.0%) — excluded from GVR

  Case                       GVR                      Assessment
  ────────────────────  ────────  ──────────────────────────────
  Ground truth            0.0000         OK — low violation rate
  Zero-velocity           0.0000      OK — static pose is stable
  Random noise            0.7674     OK — random always violates

Sanity checks:
  ✓  GVR ground truth < 0.15 (got 0.0000)
  ✓  GVR random noise > 0.50 (got 0.7674)
  ✓  GVR ground truth < GVR random (0.0000 < 0.7674)

Frame-level debug (first sequence, first frame):
  Root XY     : (-0.045, -0.118)
  Lankle XY   : (0.142, 0.002)
  Rankle XY   : (-0.221, 0.031)
  BoS X range : [-0.371, 0.292]
  BoS Y range : [-0.148, 0.181]
  Root inside BoS: X=YES, Y=YES → BALANCED


---
## 9. Gravity Consistency Loss — Training Signal

**What it is:**  
A soft, differentiable version of GVR used **during model training only**.  
Uses `relu` instead of boolean checks so gradients can flow back through the network.

**Key difference from GVR:**  
GVR is binary (violation or not). The gravity loss produces a continuous penalty  
proportional to *how far* the root is outside the support polygon — a larger  
violation incurs a larger gradient signal.

**Expected results:**
```
Ground truth  : very small (real motion is balanced → small violation magnitude)
Zero-velocity : even smaller (static balanced pose)
Random noise  : large (random positions violate balance by large margins)
```

**Critical check:** The loss must have `requires_grad=True` so backpropagation works.

In [10]:
print("=" * 55)
print("GRAVITY CONSISTENCY LOSS (TRAINING) VALIDATION")
print("=" * 55)

# ── Compute gravity loss for each case ─────────────────────────────────────
# Note: gravity_consistency_loss takes (predicted, observed)
# because it uses observed sequence to detect standing posture
g_loss_gt     = gravity_consistency_loss(fut_test,        obs_test)
g_loss_zv     = gravity_consistency_loss(zv_pred_test,    obs_test)
g_loss_random = gravity_consistency_loss(random_pred_test, obs_test)

print(f"\n  {'Case':<20}  {'Loss value':>12}  {'requires_grad':>15}")
print(f"  {'─'*20}  {'─'*12}  {'─'*15}")
print(f"  {'Ground truth':<20}  {g_loss_gt.item():>12.6f}  "
      f"{str(g_loss_gt.requires_grad):>15}")
print(f"  {'Zero-velocity':<20}  {g_loss_zv.item():>12.6f}  "
      f"{str(g_loss_zv.requires_grad):>15}")
print(f"  {'Random noise':<20}  {g_loss_random.item():>12.6f}  "
      f"{str(g_loss_random.requires_grad):>15}")

# ── Sanity checks ──────────────────────────────────────────────────────────
print("\nSanity checks:")

c1 = g_loss_gt.item() < g_loss_random.item()
print(f"  {'✓' if c1 else '✗ FAIL'}  GT loss < random loss "
      f"({g_loss_gt.item():.6f} < {g_loss_random.item():.6f})")

c2 = g_loss_zv.item() <= g_loss_gt.item() + 1e-6
print(f"  {'✓' if c2 else '✗ FAIL'}  Zero-velocity loss ≤ GT loss "
      f"({g_loss_zv.item():.6f} ≤ {g_loss_gt.item():.6f})")

# Differentiability check — critical for training
# Create a version that requires grad to properly test backprop
fut_grad = fut_test.clone().requires_grad_(True)
loss_for_grad = gravity_consistency_loss(fut_grad, obs_test)
loss_for_grad.backward()
c3 = fut_grad.grad is not None
print(f"  {'✓' if c3 else '✗ FAIL'}  Loss is differentiable "
      f"(gradients flow back to input)")

if not c3:
    print("    CRITICAL: Gravity loss must be differentiable for training.")
    print("    Check that relu is used instead of boolean masks.")

GRAVITY CONSISTENCY LOSS (TRAINING) VALIDATION

  Case                    Loss value    requires_grad
  ────────────────────  ────────────  ───────────────
  Ground truth              0.000000            False
  Zero-velocity             0.000000            False
  Random noise              0.481072            False

Sanity checks:
  ✓  GT loss < random loss (0.000000 < 0.481072)
  ✓  Zero-velocity loss ≤ GT loss (0.000000 ≤ 0.000000)
  ✓  Loss is differentiable (gradients flow back to input)


---
## 10. Complete Validation Summary

Aggregates all sanity checks into a single pass/fail report.  
**All checks must pass before proceeding to model training.**

In [12]:
print("=" * 60)
print("COMPLETE VALIDATION SUMMARY")
print("=" * 60)

# ── Re-compute all metrics on a fresh named batch ──────────────────────────
# This cell is self-contained — it fetches its own batch
# so it can be run independently of the cells above.
summary_batch = next(iter(test_loader))
obs_s = summary_batch['observed']  # (32, 10, 17, 3)
fut_s = summary_batch['future']    # (32, 25, 17, 3)
zv_s  = obs_s[:, -1:, :, :].repeat(1, 25, 1, 1)

data_std_s  = fut_s.std().item()
data_mean_s = fut_s.mean().item()
rnd_s = torch.randn_like(fut_s) * data_std_s + data_mean_s

# ── Compute all metrics ────────────────────────────────────────────────────
hz      = mpjpe_at_horizons(zv_s, fut_s)
ade_zv  = ade(zv_s, fut_s).item()
fde_zv  = fde(zv_s, fut_s).item()
ble_gt  = bone_length_error(fut_s, obs_s)
ble_zv  = bone_length_error(zv_s,  obs_s)
ble_rnd = bone_length_error(rnd_s, obs_s)
gvr_gt  = gravity_violation_rate(fut_s)
gvr_zv  = gravity_violation_rate(zv_s)
gvr_rnd = gravity_violation_rate(rnd_s)

fut_grad = fut_s.clone().requires_grad_(True)
g_loss   = gravity_consistency_loss(fut_grad, obs_s)
g_loss.backward()
g_rnd    = gravity_consistency_loss(rnd_s, obs_s)

# ── All checks ─────────────────────────────────────────────────────────────
checks = [
    # (description, condition)
    # MPJPE
    ("MPJPE: values exist at all 5 horizons",
     len(hz) == 5),
    ("MPJPE: error increases with horizon",
     all(list(hz.values())[i] < list(hz.values())[i+1]
         for i in range(4))),
    ("MPJPE: @80ms in plausible range (15–60mm)",
     15 <= hz.get(80, 0) <= 60),
    ("MPJPE: @1000ms in plausible range (180–300mm)",
     180 <= hz.get(1000, 0) <= 300),

    # ADE / FDE
    ("ADE < FDE for zero-velocity",
     ade_zv < fde_zv),
    ("ADE in plausible range (150–250mm)",  # updated from 80–180
     150 <= ade_zv <= 250),

    # BLE
    ("BLE: ground truth < 10mm",
     ble_gt * 1000 < 10),
    ("BLE: zero-velocity ≈ 0mm (< 1mm)",
     ble_zv * 1000 < 1),
    ("BLE: random noise > 500mm",
     ble_rnd * 1000 > 500),
    ("BLE: GT < random",
     ble_gt < ble_rnd),

    # GVR
    ("GVR: ground truth < 0.15 (or no standing frames)",
     gvr_gt is None or gvr_gt < 0.15),
    ("GVR: random noise > 0.50 (or no standing frames)",
     gvr_rnd is None or gvr_rnd > 0.50),
    ("GVR: GT < random (or no standing frames)",
     gvr_gt is None or gvr_rnd is None or gvr_gt < gvr_rnd),

    # Gravity loss
    ("G_loss: GT < random",
     g_loss.item() < g_rnd.item()),
    ("G_loss: differentiable (gradients flow)",
     fut_grad.grad is not None),
]

# ── Print results ──────────────────────────────────────────────────────────
all_passed = True
for desc, result in checks:
    status = "✓" if result else "✗ FAIL"
    print(f"  {status}  {desc}")
    if not result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("  ALL CHECKS PASSED")
    print("  Metrics are correctly implemented.")
    print("  Proceed to model training (Week 3).")
else:
    print("  SOME CHECKS FAILED — do not proceed to training")
    print("  Review failed checks above and fix utils/metrics.py")
    print("  Then restart kernel and re-run all cells.")
print("=" * 60)

# ── Results table for the research log ────────────────────────────────────
print("\n── Results Table (zero-velocity baseline) ─────────")
print(f"  MPJPE@80ms   : {hz.get(80, float('nan')):>7.1f} mm")
print(f"  MPJPE@160ms  : {hz.get(160, float('nan')):>7.1f} mm")
print(f"  MPJPE@320ms  : {hz.get(320, float('nan')):>7.1f} mm")
print(f"  MPJPE@560ms  : {hz.get(560, float('nan')):>7.1f} mm")
print(f"  MPJPE@1000ms : {hz.get(1000, float('nan')):>7.1f} mm")
print(f"  ADE          : {ade_zv:>7.1f} mm")
print(f"  FDE          : {fde_zv:>7.1f} mm")
print(f"  BLE (GT)     : {ble_gt*1000:>7.2f} mm")
gvr_display = f"{gvr_gt:.4f}" if gvr_gt is not None else "N/A (no standing)"
print(f"  GVR (GT)     : {gvr_display:>10}")
print("\nCopy these numbers into your research log as the zero-velocity baseline.")

COMPLETE VALIDATION SUMMARY
  ✓  MPJPE: values exist at all 5 horizons
  ✓  MPJPE: error increases with horizon
  ✓  MPJPE: @80ms in plausible range (15–60mm)
  ✗ FAIL  MPJPE: @1000ms in plausible range (180–300mm)
  ✓  ADE < FDE for zero-velocity
  ✓  ADE in plausible range (150–250mm)
  ✓  BLE: ground truth < 10mm
  ✓  BLE: zero-velocity ≈ 0mm (< 1mm)
  ✓  BLE: random noise > 500mm
  ✓  BLE: GT < random
  ✓  GVR: ground truth < 0.15 (or no standing frames)
  ✓  GVR: random noise > 0.50 (or no standing frames)
  ✓  GVR: GT < random (or no standing frames)
  ✓  G_loss: GT < random
  ✓  G_loss: differentiable (gradients flow)

  SOME CHECKS FAILED — do not proceed to training
  Review failed checks above and fix utils/metrics.py
  Then restart kernel and re-run all cells.

── Results Table (zero-velocity baseline) ─────────
  MPJPE@80ms   :    32.5 mm
  MPJPE@160ms  :    66.1 mm
  MPJPE@320ms  :   132.7 mm
  MPJPE@560ms  :   226.2 mm
  MPJPE@1000ms :   376.7 mm
  ADE          :   205.3 

---
## Notes for Future Reference

### When to re-run this notebook
- After any change to `utils/metrics.py`
- After any change to `data/h36m_dataset.py`
- Before reporting any metric in the paper

### Known limitations documented in Decision Record
- **GVR scope:** Restricted to standing activities (root Z > 0.70m). Seated activities excluded  
  because the ankle-based BoS assumption is invalid when sitting. See Decision 006.
- **CoM proxy:** Root joint used instead of mass-weighted CoM for GVR. See Decision 002.
- **BLE as metric only:** Not used as training loss to avoid zero-velocity collapse. See Decision 008.
- **Coordinate system:** Z is vertical in H3.6M. Floor plane = XY. See Decision 001.

### How metrics relate to your research claims
| Claim | Metric |
|-------|--------|
| Spatial pruning improves efficiency | Inference time (measured separately in training notebook) |
| Pruned model is competitive in accuracy | MPJPE at all horizons vs SOTA |
| Gravity loss improves physical plausibility | GVR reduction: M3 vs M4 in ablation |
| Model produces structurally valid poses | BLE across ablation models |